# COMP 352 Final Project - Spotify Predictions

### By Cameron McNamara, Bilal Adam, Maximo Babun

Requirements: 
  - There are four sections of the final project. You are expected to perform the following tasks within each section to fulfill the project requirements. Remember data science is cyclical in nature and requires multiple attempts and iterations. It is okay if your code moves between sections as you try different approaches, but at the end please try and organize your code into these sections for grading purposes.
- Data Importing and Pre-processing (100 Points)
  - Import dataset and describe characteristics such as dimensions, data types, file types, and import methods used
  - Clean, wrangle, and handle missing data, duplicate data, etc.
  - Encode any categorical variables
  - Perform feature engineering on the dataset
  - Transform data appropriately using techniques such as aggregation, normalization, and feature construction
  - Reduce redundant data and perform need based discretization
- Data Analysis and Visualization (100 Points)
  - Identify categorical, ordinal, and numerical variables within data
  - Provide measures of centrality and distribution with visualizations
  - Diagnose for correlations between variables and determine independent and dependent variables
  - Perform exploratory analysis in combination with visualization techniques to discover patterns and features of interest
  - Create visualizations that allow for the discovery of insights in the data

- Data Analytics (100 Points)
  - Determine the need for a supervised or unsupervised learning method and identify dependent and independent variables
  - Choose and provide reasoning for the selected metric or metrics employed to assess your model.
  - Train, test, cross validate, and provide performance metrics for model results
  - Try multiple different types of algorithms to determine the best model for your dataset
  - Analyze your model performance


First we must setup our environment to make sure we have all appropriate modules installed. To do this, I have provided 2 methods. The 1st, is to install all modules using a ```.yaml``` file via ```conda```. 

To do this, run:
```bash
conda env create -f env_setup/data_environment.yml
```
Then activate the environment by:
```bash
conda activate data_env
```

## Data Importing and Pre-processing <a class="anchor" id="data-importing"></a>

In [1]:
# import libraries needed
import pandas as pd

pd.set_option("display.max_columns", None)
import warnings

import branca
import folium
import geopandas as gpd
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import xgboost as xgb
from branca.element import Figure
from folium import Marker
from folium.plugins import HeatMap
from scipy.special import boxcox1p
from scipy.stats import norm, probplot, skew
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from utils.model_utils import (
    time_series_split_regression,
    StackedEnsembleCVRegressor,
)
from utils.metrics_utils import (
    compute_rmse_std,
    print_rmse_and_dates,
)

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning, module="pandas.*")
%matplotlib inline

In [3]:
## Import Data
spotify = pd.read_csv("SpotifyFeatures.csv")

#### Explore Dataset Dimensions

In [15]:
print(spotify.shape)
print("Total Observations:", spotify.shape[0])

(232725, 16)
Total Observations: 232725


In [16]:
cat_count = 0
for dtype in spotify.dtypes:
    if dtype == "object":
        cat_count = cat_count + 1

In [11]:
print("# of categorical variables:", cat_count)

numeric_vars = spotify.shape[1] - cat_count - 1
print("# of continous variables:", numeric_vars)

# of categorical variables: 7
# of continous variables: 8


#### Remove Unecessary Columns

In [9]:
spotify.head()

,genre,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
0,Movie,Henri Salvador,C'est beau de faire un Show,0BRjO6ga9RKCKjfDqeFgWV,0,0.611,0.389,99373,0.910,0.000,C#,0.3460,-1.828,Major,0.0525,166.969,4/4,0.814
1,Movie,Martin & les fées,Perdu d'avance (par Gad Elmaleh),0BjC1NfoEOOusryehmNudP,1,0.246,0.590,137373,0.737,0.000,F#,0.1510,-5.559,Minor,0.0868,174.003,4/4,0.816
2,Movie,Joseph Williams,Don't Let Me Be Lonely Tonight,0CoSDzoNIKCRs124s9uTVy,3,0.952,0.663,170267,0.131,0.000,C,0.1030,-13.879,Minor,0.0362,99.488,5/4,0.368
3,Movie,Henri Salvador,Dis-moi Monsieur Gordon Cooper,0Gc6TVm52BwZD07Ki6tIvf,0,0.703,0.240,152427,0.326,0.000,C#,0.0985,-12.178,Major,0.0395,171.758,4/4,0.227
4,Movie,Fabien Nataf,Ouverture,0IuslXpMROHdEPvSl1fTQK,4,0.950,0.331,82625,0.225,0.123,F,0.2020,-21.150,Major,0.0456,140.576,4/4,0.390


In [10]:
# Remove Artist_name and track_name - high cardinality providing no predictive power.
spotify = spotify.drop(columns=["artist_name", "track_name"])

#### Check for missing values

In [17]:
total = spotify.isnull().sum().sort_values(ascending=False)
percent = (spotify.isnull().sum() / spotify.isnull().count()).sort_values(
    ascending=False
)
missing_data = pd.concat([total, percent], axis=1, keys=["Total", "Percent"])
missing_data.head(20)

,Total,Percent
genre,0,0.0
track_id,0,0.0
popularity,0,0.0
acousticness,0,0.0
danceability,0,0.0
duration_ms,0,0.0
energy,0,0.0
instrumentalness,0,0.0
key,0,0.0
liveness,0,0.0


In [ ]:
# No Missing Values - Moving onto one-hot encoding of categorical variables

#### Check for repeats

In [ ]:
spotify["track_id"].duplicated().sum()

np.int64(55951)

In [21]:
dup_ids = spotify[spotify["track_id"].duplicated(keep=False)]
dup_ids.sort_values("track_id").head(10)


,genre,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
14492,Dance,000xQL6tZNLJzIrtIgxqSl,70,0.1310,0.748,188491,0.6270,0.00000,G,0.0852,-6.029,Major,0.0644,120.963,4/4,0.5240
110840,Pop,000xQL6tZNLJzIrtIgxqSl,70,0.1310,0.748,188491,0.6270,0.00000,G,0.0852,-6.029,Major,0.0644,120.963,4/4,0.5240
96926,Children’s Music,001gDjxhKGDSx4sMMAgS9R,57,0.0349,0.564,211789,0.8080,0.00036,C#,0.3260,-5.825,Major,0.0481,78.439,4/4,0.3650
153533,Rock,001gDjxhKGDSx4sMMAgS9R,58,0.0349,0.564,211789,0.8080,0.00036,C#,0.3260,-5.825,Major,0.0481,78.439,4/4,0.3650
65253,Folk,001ifh9Zkyc5DhK7AGQRtK,42,0.4470,0.411,395573,0.4220,0.12100,E,0.0742,-5.475,Minor,0.0459,147.465,1/4,0.3460
145387,Indie,001ifh9Zkyc5DhK7AGQRtK,42,0.4470,0.411,395573,0.4220,0.12100,E,0.0742,-5.475,Minor,0.0459,147.465,1/4,0.3460
201614,Soundtrack,002PgfoyfrOGiKch4EW8Wm,33,0.9850,0.199,46867,0.0376,0.62800,G#,0.1150,-31.142,Major,0.0457,68.167,4/4,0.0891
182413,Movie,002PgfoyfrOGiKch4EW8Wm,33,0.9850,0.199,46867,0.0376,0.62800,G#,0.1150,-31.142,Major,0.0457,68.167,4/4,0.0891
121130,Rap,002QT7AS6h1LAF5dla8D92,50,0.0469,0.830,207827,0.6530,0.00000,C#,0.1120,-5.298,Major,0.1850,123.032,4/4,0.2280
90714,Hip-Hop,002QT7AS6h1LAF5dla8D92,50,0.0469,0.830,207827,0.6530,0.00000,C#,0.1120,-5.298,Major,0.1850,123.032,4/4,0.2280


In [24]:
spotify_clean = (
    spotify.groupby("track_id", as_index=False)
           .agg({
               **{col: "first" for col in spotify.columns
                  if col not in ["track_id", "genre", "popularity"]},
               "genre": lambda x: x.mode()[0],
               "popularity": "max"
           })
)                                                                                